In [1]:
import os
os.makedirs("data/processed", exist_ok=True)
os.makedirs("data/db", exist_ok=True)
print("Folders ready")

Folders ready


In [3]:
import pandas as pd

df_nav = pd.read_csv("data/raw/02_nav_history.csv")
print(df_nav.shape)         # check row count
print(df_nav.dtypes)        # check column types
print(df_nav.isnull().sum()) # check nulls

(46000, 3)
amfi_code      int64
date          object
nav          float64
dtype: object
amfi_code    0
date         0
nav          0
dtype: int64


In [4]:
# 1. Parse date column to proper datetime
df_nav["date"] = pd.to_datetime(df_nav["date"])

# 2. Sort by fund code + date (important for ffill to work correctly)
df_nav = df_nav.sort_values(["amfi_code", "date"])

# 3. Forward-fill missing NAV values (weekends/holidays)
df_nav["nav"] = df_nav.groupby("amfi_code")["nav"].ffill()

# 4. Remove duplicate rows
df_nav = df_nav.drop_duplicates(subset=["amfi_code", "date"])

# 5. Remove rows where NAV is 0 or negative (invalid)
df_nav = df_nav[df_nav["nav"] > 0]

# 6. Compute daily return %
df_nav["daily_return_pct"] = df_nav.groupby("amfi_code")["nav"].pct_change() * 100

print(df_nav.shape)
df_nav.to_csv("data/processed/clean_nav.csv", index=False)
print("Saved clean_nav.csv")

(46000, 4)
Saved clean_nav.csv


In [5]:
df_tx = pd.read_csv("data/raw/08_investor_transactions.csv")
print(df_tx.shape)
print(df_tx["transaction_type"].unique())  # check values
print(df_tx["kyc_status"].unique())

(32778, 13)
['SIP' 'Redemption' 'Lumpsum']
['Verified' 'Pending']


In [6]:
# 1. Standardise transaction_type to title case
df_tx["transaction_type"] = df_tx["transaction_type"].str.strip().str.title()
# Should result in: SIP, Lumpsum, Redemption

# 2. Remove rows with amount <= 0
df_tx = df_tx[df_tx["amount_inr"] > 0]

# 3. Parse date
df_tx["transaction_date"] = pd.to_datetime(df_tx["transaction_date"])

# 4. Remove nulls in key columns
df_tx = df_tx.dropna(subset=["investor_id", "amfi_code", "transaction_date"])

print(df_tx.shape)
df_tx.to_csv("data/processed/clean_transactions.csv", index=False)
print("Saved clean_transactions.csv")

(32778, 13)
Saved clean_transactions.csv


In [7]:
df_perf = pd.read_csv("data/raw/07_scheme_performance.csv")
print(df_perf.dtypes)
print(df_perf.isnull().sum())

amfi_code               int64
scheme_name            object
fund_house             object
category               object
plan                   object
return_1yr_pct        float64
return_3yr_pct        float64
return_5yr_pct        float64
benchmark_3yr_pct     float64
alpha                 float64
beta                  float64
sharpe_ratio          float64
sortino_ratio         float64
std_dev_ann_pct       float64
max_drawdown_pct      float64
aum_crore               int64
expense_ratio_pct     float64
morningstar_rating      int64
risk_grade             object
dtype: object
amfi_code             0
scheme_name           0
fund_house            0
category              0
plan                  0
return_1yr_pct        0
return_3yr_pct        0
return_5yr_pct        0
benchmark_3yr_pct     0
alpha                 0
beta                  0
sharpe_ratio          0
sortino_ratio         0
std_dev_ann_pct       0
max_drawdown_pct      0
aum_crore             0
expense_ratio_pct     0
mornings

In [8]:
import numpy as np

# 1. Ensure return columns are numeric (coerce errors to NaN)
return_cols = ["return_1yr_pct", "return_3yr_pct", "return_5yr_pct",
               "sharpe_ratio", "sortino_ratio", "alpha", "beta",
               "max_drawdown_pct", "std_dev_ann_pct"]

for col in return_cols:
    if col in df_perf.columns:
        df_perf[col] = pd.to_numeric(df_perf[col], errors="coerce")

# 2. Flag negative Sharpe ratios (don't remove, just flag)
df_perf["low_sharpe_flag"] = df_perf["sharpe_ratio"] < 0

# 3. Validate expense ratio is between 0.1% and 2.5%
if "expense_ratio_pct" in df_perf.columns:
    bad = df_perf[(df_perf["expense_ratio_pct"] < 0.1) | 
                  (df_perf["expense_ratio_pct"] > 2.5)]
    print(f"Suspicious expense ratios: {len(bad)} rows")

df_perf.to_csv("data/processed/clean_schema_performance.csv", index=False)
print("Saved clean_schema_performance.csv")

Suspicious expense ratios: 0 rows
Saved clean_schema_performance.csv


In [9]:
# Map raw filename -> output filename
other_files = {
    "01_fund_master.csv": "clean_fund_master.csv",
    "03_aum_by_fund_house.csv": "clean_aum.csv",
    "04_monthly_sip_inflows.csv": "clean_sip_inflows.csv",
    "05_category_inflows.csv": "clean_category_inflows.csv",
    "06_industry_folio_count.csv": "clean_folio_count.csv",
    "09_portfolio_holdings.csv": "clean_portfolio_holdings.csv",
    "10_benchmark_indices.csv": "clean_benchmark_indices.csv",
}

for raw_name, clean_name in other_files.items():
    df = pd.read_csv(f"data/raw/{raw_name}")
    
    # Standard cleaning: strip whitespace from string columns
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].str.strip()
    
    # Drop fully empty rows
    df = df.dropna(how="all")
    
    df.to_csv(f"data/processed/{clean_name}", index=False)
    print(f"{raw_name} → {clean_name} | Shape: {df.shape}")

01_fund_master.csv → clean_fund_master.csv | Shape: (40, 15)
03_aum_by_fund_house.csv → clean_aum.csv | Shape: (90, 5)
04_monthly_sip_inflows.csv → clean_sip_inflows.csv | Shape: (48, 6)
05_category_inflows.csv → clean_category_inflows.csv | Shape: (144, 3)
06_industry_folio_count.csv → clean_folio_count.csv | Shape: (21, 6)
09_portfolio_holdings.csv → clean_portfolio_holdings.csv | Shape: (322, 8)
10_benchmark_indices.csv → clean_benchmark_indices.csv | Shape: (8050, 3)
